# Cancer Transcriptomics: Exploratory Data Analysis (EDA)

This notebook demonstrates exploratory analysis on transcriptomics data:
1. **Class balance & sample quality check**
2. **Library size distribution across conditions and batches**
3. **Principal Component Analysis (PCA)** to visualize biological separation vs. batch effects
4. **Volcano plot** of differential expression effect sizes and false discovery rates


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Paths
DATA_PATH = "../data/synthetic/synth_expression_log2cpm.csv"
META_PATH = "../data/synthetic/synth_metadata.csv"
DE_PATH = "../results/differential_expression.csv"


## 1. Load Data and Metadata


In [ ]:
expr = pd.read_csv(DATA_PATH, index_col=0)
meta = pd.read_csv(META_PATH).set_index("sample_id")

y = expr["label"]
X = expr.drop(columns=["label"])

print(f"Samples: {X.shape[0]}, Genes: {X.shape[1]}")
print(f"Class counts:\n{y.value_counts()}")


## 2. Library Size Distribution by Condition and Batch


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# By condition
normal_lib = meta[meta["label"] == 0]["library_size"]
tumour_lib = meta[meta["label"] == 1]["library_size"]
axes[0].boxplot([normal_lib, tumour_lib], tick_labels=["Normal", "Tumour"])
axes[0].set_title("Library Size by Condition")
axes[0].set_ylabel("Total Counts")

# By batch
batches = sorted(meta["batch"].unique())
batch_libs = [meta[meta["batch"] == b]["library_size"] for b in batches]
axes[1].boxplot(batch_libs, tick_labels=batches)
axes[1].set_title("Library Size by Sequencing Batch")
axes[1].set_ylabel("Total Counts")

plt.tight_layout()
plt.show()


## 3. Dimensionality Reduction: PCA (Condition vs Batch)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Coloured by tumour vs normal
colors = ["#2b5c8f" if label == 1 else "#7ea172" for label in y]
axes[0].scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.7, edgecolors="none", s=40)
axes[0].set_title(f"PCA by Condition (Var: {pca.explained_variance_ratio_[:2].sum():.1%})")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# Right: Coloured by batch
batch_map = {"batch_1": "#e41a1c", "batch_2": "#377eb8", "batch_3": "#4daf4a"}
b_colors = [batch_map.get(meta.loc[idx, "batch"], "gray") for idx in X.index]
axes[1].scatter(coords[:, 0], coords[:, 1], c=b_colors, alpha=0.7, edgecolors="none", s=40)
axes[1].set_title("PCA by Batch")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()


## 4. Differential Expression Volcano Plot


In [ ]:
if os.path.exists(DE_PATH):
    de = pd.read_csv(DE_PATH, index_col=0)
    fig, ax = plt.subplots(figsize=(7, 5))
    neg_log_fdr = -np.log10(de["fdr_bh"] + 1e-30)
    sig_mask = de["fdr_bh"] < 0.05
    
    ax.scatter(de.loc[~sig_mask, "log2_fold_change"], neg_log_fdr[~sig_mask], color="gray", alpha=0.4, label="FDR >= 0.05")
    ax.scatter(de.loc[sig_mask, "log2_fold_change"], neg_log_fdr[sig_mask], color="crimson", alpha=0.8, label="FDR < 0.05")
    ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Log2 Fold Change (Tumour - Normal)")
    ax.set_ylabel("-Log10 FDR")
    ax.set_title("Differential Expression Volcano Plot")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Run src/differential_expression.py first to generate DE results.")
